<img src="https://fedesarrollo.org.co/logo-Fede.png" alt="Fedesarrollo" style="float: right; width: 300px; height: auto;">
<img src="https://upload.wikimedia.org/wikipedia/commons/f/f9/Logo_BID_Espa%C3%B1ol.png" alt="Banco Interamericano de Desarrollo" style="float: right; width: 220px; height: auto;">

# Identificación de episodios de auge exportador

Editor: Juan Diego Heredia Niño - jheredia@fedesarrollo.org.co

Septiembre 2026

### Importar librerías

La siguiente celda marca el inicio de un nuevo bloque de trabajo independiente:

**Librerías utilizadas:**
- `pandas` y `numpy`: para manipulación y análisis de datos estructurados.
- `statsmodels`: modelos lineales generalizados y suavizamiento exponencial.
- `tqdm`: barra de avance de la identificación por producto y destino.

In [ ]:
import pandas as pd
import numpy as np

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.genmod.families import Gaussian
from statsmodels.genmod.families.links import identity
from statsmodels.tsa.holtwinters import ExponentialSmoothing

from tqdm.auto import tqdm
import unicodedata, re

### Configuración de rutas

Carga las rutas del proyecto y las opciones globales desde `config.py`, instalado como
paquete editable en la raíz del proyecto.

In [2]:
from config import raw, temp, random_state

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

### Parámetros

Fija la ventana de años y los umbrales de crecimiento que delimitan un episodio de auge
exportador, siguiendo los criterios de Freund y Pierola.

In [3]:
ventana_anios = 7  # Ventana de años para proyección
umbral_crec_abs = 0.06        # 6%
fraccion_acel = 1/3           # un tercio
umbral_acel_pp = 3            # 3 puntos porcentuales
anio_fin_datos = 2023
anios_proyeccion = ventana_anios          

# Grado máximo de polinomios opcional en la regresión
grado_polinomios = 1

# Semilla para reproducibilidad en métodos estocásticos (si aplica)
random_seed = 2025

### Lectura del panel de comercio

Carga el panel bilateral consolidado y ajusta los tipos de las columnas.

In [4]:
ruta_parquet = temp / 'exportaciones' / 'df_comercio.parquet'

# 2.1 Leer el archivo Parquet
df_trade = pd.read_parquet(ruta_parquet)


# 2.3 Asegurar tipos de datos correctos
df_trade["anio"]            = df_trade["anio"].astype(int)
df_trade["hs6_1996"]        = df_trade["hs6_1996"].astype(str)
df_trade["o_iso3"]          = df_trade["o_iso3"].astype(str)
df_trade["d_iso3"]          = df_trade["d_iso3"].astype(str)
df_trade["valor_miles_usd"] = pd.to_numeric(df_trade["valor_miles_usd"], errors="coerce")

# 2.4 Filtrar hasta el año tope parametrizado
df_trade = df_trade.loc[df_trade["anio"] <= anio_fin_datos].copy()

print("Observaciones totales:", len(df_trade))
df_trade.head()

Observaciones totales: 15575422


,anio,hs6_1996,nombre_o,o_iso3,region_o,ingreso_o,nombre_d,d_iso3,region_d,ingreso_d,valor_miles_usd,deflactor_2019
0,2004,620990,Thailand,THA,East Asia & Pacific,Upper middle income,Angola,AGO,Sub-Saharan Africa,Lower middle income,83.694,1.315
1,2004,620990,Vietnam,VNM,East Asia & Pacific,Lower middle income,Angola,AGO,Sub-Saharan Africa,Lower middle income,77.221,1.315
2,2004,621132,Brazil,BRA,Latin America & Caribbean,Upper middle income,Angola,AGO,Sub-Saharan Africa,Lower middle income,212.796,1.315
3,2004,621133,Portugal,PRT,Europe & Central Asia,High income,Angola,AGO,Sub-Saharan Africa,Lower middle income,581.756,1.315
4,2004,621142,Brazil,BRA,Latin America & Caribbean,Upper middle income,Angola,AGO,Sub-Saharan Africa,Lower middle income,67.073,1.315


### Ajustes de cobertura para México y Brasil

Corrige la asignación de los registros de esos dos países, cuya información aparece repartida
entre varias fuentes en la base original.

In [5]:
# Crear la condición
cond = (df_trade["o_iso3"] == "MEX") | (df_trade["d_iso3"] == "MEX")

# Asignar a ambas columnas en las filas que cumplen la condición
df_trade.loc[cond, ["region_o", "region_d"]] = "MEX"
df_trade.query("o_iso3 == 'MEX' or d_iso3 == 'MEX'")

,anio,hs6_1996,nombre_o,o_iso3,region_o,ingreso_o,nombre_d,d_iso3,region_d,ingreso_d,valor_miles_usd,deflactor_2019
99,2004,380820,Mexico,MEX,MEX,Upper middle income,Argentina,ARG,MEX,Upper middle income,200.401,1.315
136,2004,381230,Mexico,MEX,MEX,Upper middle income,Argentina,ARG,MEX,Upper middle income,"1,569.373",1.315
157,2004,382490,Mexico,MEX,MEX,Upper middle income,Argentina,ARG,MEX,Upper middle income,"3,835.341",1.315
187,2004,390210,Mexico,MEX,MEX,Upper middle income,Argentina,ARG,MEX,Upper middle income,323.644,1.315
212,2004,210111,Mexico,MEX,MEX,Upper middle income,Argentina,ARG,MEX,Upper middle income,368.532,1.315
...,...,...,...,...,...,...,...,...,...,...,...,...
15574333,1997,950390,Mexico,MEX,MEX,Upper middle income,South Africa,ZAF,MEX,Upper middle income,479.006,1.500
15574418,1997,870899,Mexico,MEX,MEX,Upper middle income,Venezuela,VEN,MEX,Upper middle income,"2,456.128",1.500
15574434,1997,880211,Mexico,MEX,MEX,Upper middle income,Venezuela,VEN,MEX,Upper middle income,523.040,1.500
15574454,1997,900653,Mexico,MEX,MEX,Upper middle income,Venezuela,VEN,MEX,Upper middle income,522.427,1.500


In [6]:
# Crear la condición
cond = (df_trade["o_iso3"] == "BRA") | (df_trade["d_iso3"] == "BRA")

# Asignar a ambas columnas en las filas que cumplen la condición
df_trade.loc[cond, ["region_o", "region_d"]] = "BRA"
df_trade.query("o_iso3 == 'BRA' or d_iso3 == 'BRA'")

,anio,hs6_1996,nombre_o,o_iso3,region_o,ingreso_o,nombre_d,d_iso3,region_d,ingreso_d,valor_miles_usd,deflactor_2019
2,2004,621132,Brazil,BRA,BRA,Upper middle income,Angola,AGO,BRA,Lower middle income,212.796,1.315
4,2004,621142,Brazil,BRA,BRA,Upper middle income,Angola,AGO,BRA,Lower middle income,67.073,1.315
6,2004,621210,Brazil,BRA,BRA,Upper middle income,Angola,AGO,BRA,Lower middle income,108.410,1.315
7,2004,621290,Brazil,BRA,BRA,Upper middle income,Angola,AGO,BRA,Lower middle income,91.890,1.315
8,2004,630260,Brazil,BRA,BRA,Upper middle income,Angola,AGO,BRA,Lower middle income,350.419,1.315
...,...,...,...,...,...,...,...,...,...,...,...,...
15574945,1997,870899,Brazil,BRA,BRA,Upper middle income,Sweden,SWE,BRA,High income,"1,827.382",1.500
15575093,1997,732020,Brazil,BRA,BRA,Upper middle income,South Africa,ZAF,BRA,Upper middle income,234.959,1.500
15575277,1997,820900,Brazil,BRA,BRA,Upper middle income,South Africa,ZAF,BRA,Upper middle income,161.938,1.500
15575308,1997,841899,Brazil,BRA,BRA,Upper middle income,South Africa,ZAF,BRA,Upper middle income,587.274,1.500


### Productos priorizados

Integra al panel la lista de bienes con oportunidad identificada.

In [7]:
ruta_productos = temp / 'exportaciones' / 'df_productos.parquet'

# 3.1 Leer la tabla de productos
df_prod = pd.read_parquet(ruta_productos)

# 3.3 Asegurar tipos de datos correctos
df_prod["hs6_1996"] = df_prod["hs6_1996"].astype(str)

# 3.4 Mantener solo las columnas necesarias
df_prod = df_prod[["hs6_1996", "tipo_oportunidad"]].copy()

# 3.5 Vista rápida
print("Productos:", len(df_prod))
df_prod.head()

Productos: 809


,hs6_1996,tipo_oportunidad
0,020130,Oportunidades a profundizar
1,020312,Oportunidades a la vista
2,020319,Oportunidades a profundizar
3,020630,Oportunidades a la vista
4,021011,Oportunidades de largo alcance


### Controles gravitacionales

Suma las variables de control, imputa la similitud cultural faltante con el promedio de la
pareja de regiones y descarta las observaciones incompletas.

In [8]:
ruta_controles = temp / 'exportaciones' / 'df_controles.parquet'

# 4.1 Leer el archivo Parquet de controles
df_ctrl = pd.read_parquet(ruta_controles)

list_ctrl = [x for x in df_ctrl.columns.to_list() if x not in ['o_iso3','d_iso3','anio']] + [
    #'industria',
    #'tipo_oportunidad'
    ]

# 4.4 Vista rápida
print("Observaciones en controles:", len(df_ctrl))

# 4.5 Unir controles con el DataFrame principal
cols_key = ["o_iso3", "d_iso3", "anio"]
df_trade = df_trade.merge(df_ctrl, on=cols_key, how="left").merge(df_prod, on="hs6_1996", how="left")

print("Panel principal con controles:", df_trade.shape)
df_trade.head()

Observaciones en controles: 338065


Panel principal con controles: (15575422, 17)


,anio,hs6_1996,nombre_o,o_iso3,region_o,ingreso_o,nombre_d,d_iso3,region_d,ingreso_d,valor_miles_usd,deflactor_2019,food_index,tierra_cultivable_o,distancia,cultura_similaridad,tipo_oportunidad
0,2004,620990,Thailand,THA,East Asia & Pacific,Upper middle income,Angola,AGO,Sub-Saharan Africa,Lower middle income,83.694,1.315,77.543,NaN,"9,947.915",43.800,Oportunidades a profundizar
1,2004,620990,Vietnam,VNM,East Asia & Pacific,Lower middle income,Angola,AGO,Sub-Saharan Africa,Lower middle income,77.221,1.315,77.543,NaN,"10,627.528",48.600,Oportunidades a profundizar
2,2004,621132,Brazil,BRA,BRA,Upper middle income,Angola,AGO,BRA,Lower middle income,212.796,1.315,77.543,7.020,"6,663.716",62.800,Oportunidades a profundizar
3,2004,621133,Portugal,PRT,Europe & Central Asia,High income,Angola,AGO,Sub-Saharan Africa,Lower middle income,581.756,1.315,77.543,NaN,"5,771.917",51.700,Oportunidades a profundizar
4,2004,621142,Brazil,BRA,BRA,Upper middle income,Angola,AGO,BRA,Lower middle income,67.073,1.315,77.543,7.020,"6,663.716",62.800,Oportunidades a profundizar


In [9]:
df_inputacion_cultura = df_trade[['o_iso3','region_o','d_iso3','region_d','cultura_similaridad']].dropna().drop_duplicates().groupby(['region_o','region_d'])['cultura_similaridad'].mean().round(3).reset_index()
df_inputacion_cultura

,region_o,region_d,cultura_similaridad
0,BRA,BRA,55.046
1,East Asia & Pacific,East Asia & Pacific,52.851
2,East Asia & Pacific,Europe & Central Asia,46.896
3,East Asia & Pacific,Latin America & Caribbean,48.938
4,East Asia & Pacific,Middle East & North Africa,43.806
5,East Asia & Pacific,North America,44.680
6,East Asia & Pacific,South Asia,47.777
7,East Asia & Pacific,Sub-Saharan Africa,44.192
8,Europe & Central Asia,East Asia & Pacific,47.044
9,Europe & Central Asia,Europe & Central Asia,65.214


In [10]:
df_trade = df_trade.merge(df_inputacion_cultura, left_on=['region_o', 'region_d'], right_on=['region_o', 'region_d'], how='left')
df_trade['cultura_similaridad'] = df_trade['cultura_similaridad_x'].fillna(df_trade['cultura_similaridad_y'])
df_trade.drop(columns=['cultura_similaridad_x', 'cultura_similaridad_y','tierra_cultivable_o'], inplace=True)

In [11]:
# Identificar variables con al menos un valor nulo
null_vars = df_trade.columns[df_trade.isnull().any()]

for var in null_vars:
    print(f"Paises y productos donde {var} es nulo")
    subset = df_trade[df_trade[var].isnull()][['nombre_d', 'nombre_o', 'hs6_1996']]
    display(subset.drop_duplicates().reset_index(drop=True))

df_trade = df_trade.dropna()

Paises y productos donde distancia es nulo


,nombre_d,nombre_o,hs6_1996
0,Spain,Tokelau,300490
1,Bolivia,Tokelau,721621
2,Czech Republic,Tokelau,620342
3,Czech Republic,Tokelau,611020
4,France,Tokelau,732690
5,Czech Republic,Tokelau,610910
6,Czech Republic,Tokelau,870322
7,Czech Republic,Tokelau,870390
8,Djibouti,Tokelau,721391
9,Malaysia,Tokelau,850431


Paises y productos donde cultura_similaridad es nulo


,nombre_d,nombre_o,hs6_1996
0,Andorra,Indonesia,611020
1,Andorra,Italy,611020
2,Andorra,Malaysia,611020
3,Andorra,Netherlands,611020
4,Andorra,Thailand,611020
...,...,...,...
3453,Andorra,Switzerland,180610
3454,Andorra,Faeroe Islands,220890
3455,Andorra,United States,841989
3456,Andorra,Italy,845969


### Participaciones de mercado

Calcula el total importado de cada producto por destino y año, y la participación de cada
origen en ese mercado.

In [12]:
# (1) Calculamos el total que se importó del producto en cada año. 
df_trade["total_prod_year"] = (
    df_trade
      .groupby(["hs6_1996",'d_iso3', "anio"])["valor_miles_usd"]
      .transform("sum")
) 

# (2) Participación del país de origen en el mercado del destino para ese producto-año.
df_trade["share"] = (
    df_trade["valor_miles_usd"] / df_trade["total_prod_year"]
) 

### Continuidad de las series

Define la verificación de continuidad temporal, requisito para aplicar los criterios de
identificación de episodios.

In [13]:
def es_serie_continua(lista_anios):
    lista_anios = sorted(set(lista_anios))  # ordenar y eliminar duplicados
    rango_esperado = list(range(min(lista_anios), max(lista_anios) + 1))
    return lista_anios == rango_esperado

### Identificación de los episodios

Aplica los cinco criterios que definen un auge exportador y construye la base de episodios con
sus variables explicativas.

In [ ]:
(temp / 'exportaciones').mkdir(parents=True, exist_ok=True)

archivo_salida = temp / 'exportaciones' / 'df_surge_vars_6.parquet'

surge_vars = list()

for (origen,destino,producto), df_sub in df_trade.drop(columns=['food_index', 'distancia', 'tipo_oportunidad', 'cultura_similaridad','region_d', 'ingreso_d', 'region_o', 'ingreso_o']).groupby(["o_iso3", "d_iso3", "hs6_1996"]):
    
    df_prueba = df_sub.sort_values("anio")
    df_prueba.set_index('anio', inplace=True)

    max_year = df_prueba['share'].idxmax()
    if df_prueba[df_prueba.index < max_year].empty == False:
        min_year = df_prueba[df_prueba.index < max_year]['share'].idxmin()
        

        df_prueba['anio_max'] = max_year
        df_prueba['anio_min'] = min_year

        df_prueba.reset_index(inplace=True)

        df_prueba['periodo'] = np.where(df_prueba['anio'] < min_year, 'previo',np.where(df_prueba['anio'] > max_year, 'posterior', 'actual'))

        if es_serie_continua(df_prueba.query("periodo == 'actual'")['anio'].to_list()) == True:
            print(f"Procesando: {origen} - {destino} - {producto}")
            # Criterio 1
            df_prueba['g%'] = df_prueba['valor_miles_usd'].pct_change()
            df_prueba['g%_mean'] = df_prueba.groupby('periodo')['g%'].transform('mean')
            if df_prueba[['periodo','g%_mean']].drop_duplicates().set_index('periodo').T.reset_index(drop=True)['actual'][0] >= umbral_crec_abs:
                df_prueba['criterio1'] = True
            else:
                df_prueba['criterio1'] = False
            df_prueba.drop(columns=['g%','g%_mean'], inplace=True)

            # Criterio 2
            if 'anterior' in df_prueba['periodo'].unique().tolist():
                g_porc_promedio = df_prueba[['periodo','g%_mean']].drop_duplicates().set_index('periodo').T.reset_index(drop=True)
                if (g_porc_promedio['actual'][0] >= g_porc_promedio['anterior'][0] * (1+fraccion_acel) and g_porc_promedio['actual'][0] >= (g_porc_promedio['anterior'][0] + 0.03)):
                    df_prueba['criterio2'] = True
                else:
                    df_prueba['criterio2'] = False
            else:
                df_prueba['criterio2'] = True

            # Criterio 3
            if 'anterior' in df_prueba['periodo'].unique().tolist():
                list_5_mercados_actual = list()

                for año in df_prueba.query("periodo == 'actual'")['anio'].unique().tolist():
                    share_5 = df_trade.query("o_iso3 == @origen and anio == @año and hs6_1996 == @producto").sort_values('share', ascending=False).head(5)['share'].mean()
                    list_5_mercados_actual.append(share_5)

                actual_mean = np.mean(list_5_mercados_actual)

                list_5_mercados_anterior = list()

                for año in df_prueba.query("periodo == 'anterior'")['anio'].unique().tolist():
                    share_5 = df_trade.query("o_iso3 == @origen and anio == @año and hs6_1996 == @producto").sort_values('share', ascending=False).head(5)['share'].mean()
                    list_5_mercados_anterior.append(share_5)

                anterior_mean = np.mean(list_5_mercados_anterior)
                if actual_mean > anterior_mean:
                    df_prueba['criterio3'] = True
                else:
                    df_prueba['criterio3'] = False
            else:
                df_prueba['criterio3'] = True

            # Criterio 4
            if 'anterior' in df_prueba['periodo'].unique().tolist():
                if df_prueba.groupby('periodo')['valor_miles_usd'].max().reset_index().set_index('periodo').T.reset_index(drop=True)['anterior'][0] < df_prueba.groupby('periodo')['valor_miles_usd'].min().reset_index().set_index('periodo').T.reset_index(drop=True)['actual'][0]:
                    df_prueba['criterio4'] = True
                else:
                    df_prueba['criterio4'] = False
            else:
                df_prueba['criterio4'] = True

            # Criterio 5
            if 'anterior' in df_prueba['periodo'].unique().tolist():
                idx_max_share = df_prueba[df_prueba["periodo"] == "actual"]["valor_miles_usd"].idxmax()

                df_filtrado = df_prueba.drop(index=idx_max_share).copy()

                df_filtrado['g%'] = df_filtrado['valor_miles_usd'].pct_change()
                df_filtrado['g%_mean'] = df_filtrado.groupby('periodo')['g%'].transform('mean')
                if df_filtrado[['periodo','g%_mean']].drop_duplicates().set_index('periodo').T.reset_index(drop=True)['actual'][0] > df_filtrado[['periodo','g%_mean']].drop_duplicates().set_index('periodo').T.reset_index(drop=True)['anterior'][0]:
                    df_prueba['criterio5'] = True
                else:
                    df_prueba['criterio5'] = False
                df_prueba.drop(columns=['g%','g%_mean'], inplace=True)
            else:
                df_prueba['criterio5'] = True

            df_prueba_ep = df_prueba[['hs6_1996', 'o_iso3', 'd_iso3', 'anio_min', 'anio_max', 'criterio1', 'criterio2', 'criterio3', 'criterio4', 'criterio5']].drop_duplicates().reset_index(drop=True)
            if df_prueba_ep[['criterio1', 'criterio2', 'criterio3', 'criterio4', 'criterio5']].sum(axis=1)[0] == 5:
                surge_vars.append(
                    {
                        "o_iso3": df_prueba['o_iso3'].drop_duplicates().reset_index(drop=True)[0],
                        "d_iso3": df_prueba['d_iso3'].drop_duplicates().reset_index(drop=True)[0],
                        "hs6_1996": df_prueba['hs6_1996'].drop_duplicates().reset_index(drop=True)[0],
                        "surge_start": df_prueba.query("anio == @min_year")['anio'].drop_duplicates().reset_index(drop=True)[0],
                        "surge_end": df_prueba.query("anio == @max_year")['anio'].drop_duplicates().reset_index(drop=True)[0],
                        "x_start": df_prueba.query("anio == @min_year")['valor_miles_usd'].drop_duplicates().reset_index(drop=True)[0],
                        "x_end": df_prueba.query("anio == @max_year")['valor_miles_usd'].drop_duplicates().reset_index(drop=True)[0],
                        "m_start": df_prueba.query("anio == @min_year")['total_prod_year'].drop_duplicates().reset_index(drop=True)[0],
                        "m_end": df_prueba.query("anio == @max_year")['total_prod_year'].drop_duplicates().reset_index(drop=True)[0],
                        "share_start": df_prueba.query("anio == @min_year")['share'].drop_duplicates().reset_index(drop=True)[0]
                    }    
                )

df_surge_vars = pd.DataFrame(surge_vars)
df_surge_vars.to_parquet(archivo_salida, index=False)